# Plan Hang Analysis

Diagnose a **hung or very slow** `terraform plan` from **`TF_LOG=json`** trace output.

Looks for:

- **Graph wait loops** — `dag/walk: vertex ... is waiting for ...` (spinning in the graph)
- **Stuck refresh** — refresh started but not completing
- **Vertex churn** — graph vertices revisited repeatedly
- **SDK retry / 404 loops** — rate limits or missing Genesys Cloud objects

```bash
export TF_LOG=json
export TF_LOG_PATH=plan-hang.log
# optional: GENESYSCLOUD_SDK_DEBUG=true GENESYSCLOUD_SDK_DEBUG_FORMAT=Json
terraform plan
export TERRAFORM_LOG_PATH=plan-hang.log
```

For completed plans use `plan/log-analysis.ipynb`. Run `whatisit.ipynb` first if unsure.


In [ ]:
import sys
from pathlib import Path

_nb_root = Path.cwd()
if (_nb_root.parent / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root.parent))
elif (_nb_root / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root))

import notebook_setup

notebook_setup.setup()

import matplotlib.pyplot as plt
import pandas as pd

import commonlib.config as cfg
import commonlib.prep_hang_data as hang
import commonlib.prep_plan_log_data as prep_plan_log_data


In [ ]:
TAIL_MINUTES = 5
MIN_REPEAT = 3
WORKFLOW = "plan"

c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)

classification, records, counters = hang.load_hang_scan(c.TERRAFORM_LOG_PATH, tail_minutes=TAIL_MINUTES)
if not records:
    raise ValueError("No JSON log lines found. Capture with TF_LOG=json and run whatisit.ipynb first.")

normalized_records = prep_plan_log_data.normalize_records(records)

if not classification.is_plan and not counters.dag_waits and not counters.refresh_starts:
    raise ValueError(
        "No plan trace activity found. Use plan/hang-analysis.ipynb on a TF_LOG plan capture, "
        "or plan/output-analysis.ipynb for terraform plan -json UI output."
    )

summary = hang.hang_summary_for_workflow(
    counters,
    tail_minutes=TAIL_MINUTES,
    workflow=WORKFLOW,
    min_count=MIN_REPEAT,
    classification=classification,
)


## Summary

In [ ]:
print(f"Parsed lines: {summary['parsed_lines']:,}")
if summary['duration_minutes'] is not None:
    print(f"Log span: {summary['duration_minutes']:.1f} minutes")
if summary['first_timestamp']:
    print(f"Time range: {summary['first_timestamp']} → {summary['last_timestamp']}")
print(f"Tail window: last {summary['tail_minutes']:.0f} minutes")
print()
print(f"Primary suspect: {summary['primary_summary']}")
if summary['primary_detail']:
    print(f"  {summary['primary_detail']}")
print()
print(
    f"Plan: {summary['dag_wait_pairs']:,} dag wait pairs, "
    f"{summary['sdk_retry_endpoints']:,} retry endpoints, "
    f"{summary['sdk_404_endpoints']:,} 404 endpoints"
)
if summary['sdk_status_codes']:
    print(f"SDK status codes: {summary['sdk_status_codes']}")


## Ranked hang suspects

In [ ]:
pd.DataFrame([
    {"category": v.category, "summary": v.summary, "detail": v.detail, "score": v.score}
    for v in summary["verdicts"]
]) if summary["verdicts"] else "No strong hang patterns detected (try lowering MIN_REPEAT)."


## Graph wait loops (spinning)

Resources Terraform keeps waiting on.

In [ ]:
df_blocked = hang.blocked_resources_dataframe(counters, min_count=MIN_REPEAT)
df_blocked.head(20) if not df_blocked.empty else "No repeated dag/walk wait lines found."


In [ ]:
if not df_blocked.empty:
    plot_df = df_blocked.head(15).sort_values("wait_messages")
    plot_df = plot_df.assign(
        label=plot_df["waiting_for"].str.replace(r"^module\.", "", regex=True).str.slice(0, 60)
    )
    plt.figure(figsize=(12, 6))
    plt.barh(plot_df["label"], plot_df["wait_messages"], color="tab:orange")
    plt.xlabel("dag/walk wait lines")
    plt.title("Top resources the graph is waiting on")
    plt.tight_layout()


## Stuck refresh and vertex churn

In [ ]:
hang.refresh_imbalance_dataframe(counters, min_gap=2).head(20)


In [ ]:
hang.vertex_churn_dataframe(counters, min_count=10).head(20)


## SDK retry storms

Endpoints returning `invocation_retry_after` — rate limits or SDK backoff.

In [ ]:
df_retries = hang.sdk_retry_dataframe(counters, min_count=MIN_REPEAT)
df_retries.head(20) if not df_retries.empty else "No SDK retry storms found."


In [ ]:
if not df_retries.empty:
    plot_df = df_retries.head(15).sort_values("retry_responses")
    plt.figure(figsize=(12, 6))
    plt.barh(plot_df["method_url"], plot_df["retry_responses"], color="tab:red")
    plt.xlabel("responses with retry_after")
    plt.title("SDK retry storms")
    plt.tight_layout()


## SDK 404 not found

Missing Genesys Cloud objects — often causes endless retry.

In [ ]:
df_404 = hang.sdk_not_found_dataframe(counters, min_count=2)
df_404.head(20) if not df_404.empty else "No repeated SDK 404 responses found."


## Tail activity (last few minutes)

What the log was doing right before capture.

In [ ]:
hang.tail_messages_dataframe(counters, top_n=25)
